# FCN


## 0. 基本认识 FCN Overview

FCN（Fully Convolutional Network，全卷积网络）是 Long、Shelhamer 和 Darrell 提出的语义分割模型，对应论文是 *Fully Convolutional Networks for Semantic Segmentation*。它把用于图像分类的卷积网络改造成能够输出像素级类别预测的网络。

FCN 关注的问题是：分类网络只给出整张图像的一个类别，怎样让它保留空间位置，并判断每个像素属于什么类别。

核心背景可以概括为：

- 图像分类的输出通常是一个类别，例如“这是一只猫”。
- 语义分割的输出是和输入图像对应的类别图，例如每个像素被标为背景、猫、道路或建筑。
- 分类网络中的全连接层会固定输入尺寸，并丢失大部分空间位置关系。
- FCN 用卷积层替代全连接层，并通过上采样把低分辨率预测恢复到输入图像大小。
- FCN-32s、FCN-16s 和 FCN-8s 通过不同层级的特征融合，逐步改善分割边界。

简单理解：FCN 先用深层网络理解“图中有什么”，再把粗糙的类别图放大，并结合浅层的位置信息，得到每个像素的分类结果。


---


## 1. 语义分割任务 Semantic Segmentation

### 1.1 定义

语义分割（semantic segmentation）是像素级分类任务。给定一张输入图像，模型要为每个像素预测一个语义类别。

若输入图像为：

$$
x \in \mathbb{R}^{3 \times H \times W}
$$

共有 $C$ 个类别，模型输出的 logits 通常为：

$$
z = f(x) \in \mathbb{R}^{C \times H \times W}
$$

其中：

- $H, W$：输入图像的高和宽。
- $C$：类别数，通常包含背景类。
- $z_{c,h,w}$：位置 $(h,w)$ 属于第 $c$ 类的原始得分。
- 最终类别是该像素在类别维度上的最大得分对应的类别。

### 1.2 和分类、检测的区别

| 任务 | 输出 | 回答的问题 | 常见指标 |
| --- | --- | --- | --- |
| 图像分类 Classification | 整张图一个或多个类别 | 图里有什么 | Accuracy |
| 目标检测 Detection | 类别和边界框 | 物体在哪里 | mAP |
| 语义分割 Semantic Segmentation | 每个像素的类别 | 每个位置是什么 | mIoU、Pixel Accuracy |

语义分割不区分同类实例。例如图中有两辆车，语义分割会把它们都标为“车”；要区分两辆不同的车，需要实例分割（instance segmentation）。

### 1.3 为什么分割更困难

分类只需知道某种物体是否存在；分割还需要准确保留边界和位置。深层特征语义更强，但尺寸较小；浅层特征尺寸较大，但语义较弱。如何同时利用两者是 FCN 的关键问题。


---


## 2. 全卷积思想 Fully Convolutional Design

### 2.1 为什么分类网络不能直接做分割

以 VGG 为例，卷积和池化层会不断下采样，最后的全连接层把特征图展平为一个向量，用于整图分类。这个过程有两个问题：

- 全连接层依赖固定长度输入，因此难以自然处理任意尺寸图像。
- 展平后不再保留每个特征对应的二维位置，不能直接还原像素位置。

### 2.2 用卷积替代全连接层

FCN 把 VGG 分类器中的全连接层转换为卷积层：

| VGG 分类层 | FCN 中的等价层 | 作用 |
| --- | --- | --- |
| fc6 | $7 \times 7$ 卷积 | 在局部特征图上提取高层语义 |
| fc7 | $1 \times 1$ 卷积 | 继续做通道变换 |
| fc8 | $1 \times 1$ 卷积 | 输出 $C$ 个类别得分图 |

假设某层特征图为 $(N, D, h, w)$，其中 $N$ 是 batch size，$D$ 是通道数。用 $1 \times 1$ 卷积映射到 $C$ 类后，输出为：

$$
(N, D, h, w) \rightarrow (N, C, h, w)
$$

这张 $(h,w)$ 的得分图叫作 score map。它的每个位置都对应输入图像中的一个感受野，而不是只对应整张图。

### 2.3 全卷积的特点

- 网络中不再需要固定尺寸的全连接输入，因此可以接受不同大小的图像。实际训练时仍常把一个 batch 内的图像处理到相同尺寸。
- 卷积在空间位置共享参数，能一次输出整张图的预测，避免逐像素重复裁剪和推理。
- 输出仍然比输入小，因为池化或 stride 卷积已使特征图下采样；后续必须上采样。


---


## 3. 网络结构 Network Architecture

### 3.1 主线结构

原始 FCN 常以 VGG-16 作为 backbone。整体数据流可以概括为：

Input -> VGG 卷积和池化主干 -> 将 fc6、fc7 转为卷积层 -> 1x1 Conv 生成类别 score map -> 上采样和跳跃连接融合 -> 与输入同尺寸的分割 logits。

VGG-16 中 5 次最大池化会使空间尺寸约缩小为原图的 $1/32$。例如输入为 $224 \times 224$，最后的粗粒度得分图大约为 $7 \times 7$，因此只依赖最后一层的预测边界会很粗糙。

### 3.2 编码器和解码器

| 部分 | 主要操作 | 输出特点 |
| --- | --- | --- |
| 编码器 Encoder | 卷积、池化、下采样 | 尺寸变小，语义和感受野变大 |
| 类别预测层 | $1 \times 1$ 卷积 | 每个空间位置输出 $C$ 类得分 |
| 解码器 Decoder | 转置卷积上采样、特征融合 | 尺寸恢复，定位信息增强 |

FCN 的编码器和解码器并不像后来的 U-Net 那样完全对称。它的重点是从分类 backbone 的多层特征中取出 score map，再进行逐级融合和放大。

### 3.3 FCN-32s、FCN-16s 和 FCN-8s

| 模型 | 融合特征 | 上采样过程 | 特点 |
| --- | --- | --- | --- |
| FCN-32s | 仅 conv7 的 score map | 一次 $32\times$ 上采样 | 结构最简单，结果最粗糙 |
| FCN-16s | conv7 和 pool4 | 先 $2\times$，融合后再 $16\times$ | 改善部分边界 |
| FCN-8s | conv7、pool4 和 pool3 | 逐步融合后再 $8\times$ | 定位更好，经典 FCN 版本 |

名称中的 s 表示最终输出中相邻预测位置在输入图像上相隔的步长（stride）。步长越小，通常空间细节越好，但计算和显存开销也会增加。


---


## 4. 上采样和跳跃连接 Upsampling and Skip Connection

### 4.1 为什么需要上采样

下采样让网络拥有更大的感受野和更强的语义抽象能力，但会降低空间分辨率。分割标签与输入图像同样大，所以必须把 score map 从 $(h,w)$ 恢复到 $(H,W)$。

### 4.2 双线性插值和转置卷积

| 方法 | 是否学习参数 | 特点 | 常见用途 |
| --- | --- | --- | --- |
| 双线性插值 Bilinear Interpolation | 否 | 固定规则，平滑、简单 | 调整尺寸、基线实现 |
| 转置卷积 Transposed Convolution | 是 | 学习上采样权重 | FCN 原论文的可学习上采样 |

转置卷积不是“卷积的数学逆运算”，而是一种能够扩大空间尺寸的可学习卷积操作。其输出尺寸为：

$$
O = (I - 1) \times S - 2P + K + \text{output\_padding}
$$

其中 $I$ 是输入尺寸，$S$ 是 stride，$P$ 是 padding，$K$ 是卷积核大小，$O$ 是输出尺寸。

### 4.3 跳跃连接 Skip Connection

FCN-8s 的融合过程可以理解为：

1. 将最深层 conv7 的 score map 上采样 $2\times$。
2. 与 pool4 的 score map 做逐元素相加。
3. 将融合结果继续上采样 $2\times$。
4. 与 pool3 的 score map 做逐元素相加。
5. 最后上采样 $8\times$，得到原图大小的预测。

深层特征提供类别语义，pool3、pool4 等浅层特征保留更多位置和边界信息。相加前两个 score map 的 batch、类别通道、高和宽必须完全一致；如果通道数不同，先使用 $1 \times 1$ 卷积投影到 $C$ 类。

### 4.4 初始化和棋盘伪影

FCN 论文中常把上采样层初始化为双线性插值权重，再通过训练进行微调。转置卷积的 kernel size、stride 和 padding 组合不合适时，输出可能出现棋盘伪影。实战中也常用“插值 + 普通卷积”替代，并在融合前显式检查尺寸。


---


## 5. 损失函数和评价指标 Loss and Metrics

### 5.1 像素级交叉熵

多类别语义分割常使用像素级交叉熵。对于像素 $(h,w)$，真实类别为 $y_{h,w}$，损失为：

$$
\mathcal{L} = -\frac{1}{HW}\sum_{h=1}^{H}\sum_{w=1}^{W}\log p_{y_{h,w},h,w}
$$

其中 $p$ 是 logits 在类别维度经过 softmax 后的概率。PyTorch 的 CrossEntropyLoss 接收原始 logits，因此模型末尾不要先做 softmax。

典型张量形状为：

- 模型输出：$(N, C, H, W)$，数据类型为浮点数。
- 分割标签：$(N, H, W)$，每个位置是 $0$ 到 $C-1$ 的类别索引，数据类型通常为 long。
- 标签不是 $(N, C, H, W)$ 的 one-hot 张量，除非使用的损失函数明确要求 one-hot 标签。

### 5.2 Pixel Accuracy

像素准确率统计预测正确的像素比例：

$$
Pixel\ Accuracy = \frac{\text{预测正确像素数}}{\text{全部有效像素数}}
$$

类别严重不均衡时，这个指标容易偏高。例如大部分像素都是背景，模型只预测背景也可能得到较高准确率。

### 5.3 IoU 和 mIoU

对某一类别，交并比（Intersection over Union）为：

$$
IoU = \frac{TP}{TP + FP + FN}
$$

mIoU 是所有类别 IoU 的平均值：

$$
mIoU = \frac{1}{C}\sum_{c=1}^{C} IoU_c
$$

mIoU 同时关注漏分和误分，通常比 Pixel Accuracy 更适合评估语义分割质量。计算时要明确背景类是否计入平均，以及是否忽略 ignore index。


---


## 6. PyTorch 实现思路 PyTorch Implementation

当前 FCN 目录中只有本学习笔记，没有配套模型和训练脚本。下面按常见 PyTorch 接口整理，不对应某个本地源码文件。

### 6.1 模型输出约定

设输入图像为 $(N, 3, H, W)$。backbone 输出多层特征，例如 pool3、pool4、conv7；每层先经 $1 \times 1$ 卷积变为 $C$ 个类别通道，再用上采样和相加融合。最终输出必须恢复为：

$$
(N, C, H, W)
$$

训练时可把最终输出直接送入 CrossEntropyLoss。推理时在类别维度取 argmax，得到 $(N, H, W)$ 的类别图。

### 6.2 一个前向流程

1. 输入图像经过 VGG 或其他 CNN backbone，保存多个阶段的特征。
2. 用 $1 \times 1$ 卷积把选中的特征映射为类别 score map。
3. 对深层 score map 上采样，使其和浅层 score map 尺寸一致。
4. 将同尺寸 score map 逐元素相加。
5. 把融合结果上采样到标签或输入图像的大小。
6. 计算像素级交叉熵，反向传播更新网络参数。

### 6.3 数据处理

图像和 mask 必须使用相同的几何变换。例如随机裁剪、翻转、缩放必须对两者同步执行；否则图像中的道路位置与标签中的道路位置会错开。

对 mask 使用最近邻插值（nearest interpolation），不能使用双线性插值。双线性插值会把离散类别编号混合成无意义的小数或新数值。

### 6.4 预训练权重

FCN 常加载在 ImageNet 上预训练的 VGG 主干，以获得基础的边缘、纹理和物体特征。需要注意：分类器转卷积层时，权重形状要正确重排；类别预测层和上采样层通常需要针对当前数据集重新训练。


---


## 7. 注意点 Common Pitfalls

### 7.1 输出尺寸和标签尺寸不一致

FCN 中连续池化、转置卷积和输入尺寸不能整除 stride 都可能造成一两个像素的偏差。计算损失前必须保证输出 logits 的空间尺寸与标签一致。必要时统一把 logits 插值到标签大小，但应先定位是哪个层的参数导致偏差。

### 7.2 跳跃连接相加前形状必须一致

FCN 使用逐元素相加，而不是通道拼接。因此两个张量的形状必须完全相同，尤其是类别通道 $C$ 和空间尺寸 $(H,W)$。不要把 U-Net 常见的 concat 融合方式直接套到原始 FCN 中。

### 7.3 不要错误处理分割标签

标签应保留离散类别编号。常见错误包括把标签除以 255 后直接用于多类交叉熵、对标签做归一化、使用双线性缩放 mask，或把调色板 PNG 的显示颜色误当作类别索引。

### 7.4 不要在模型末尾手动加 Softmax

使用 CrossEntropyLoss 时，模型应输出 logits。交叉熵内部已完成数值稳定的 softmax 相关计算。只有在可视化概率或进行概率阈值判断时，才单独计算 softmax。

### 7.5 类别不均衡和 ignore index

背景像素常远多于前景像素，单用普通交叉熵可能让模型偏向背景。可以按任务情况使用类别权重、Dice Loss 或 Focal Loss。数据集中用于填充、边界不确定或未标注的像素应设置 ignore index，并在损失和指标计算中一致忽略。

### 7.6 原始 FCN 和后续模型不要混淆

| 模型 | 主要融合方式 | 典型特点 |
| --- | --- | --- |
| FCN | 上采样加跳跃连接 | 开创端到端全卷积分割 |
| U-Net | 对称编码器解码器，concat skip connection | 小数据和医学图像中常见 |
| DeepLab | 空洞卷积、ASPP 等 | 扩大感受野并兼顾分辨率 |

学习 FCN 时，应先掌握全连接转卷积、score map、可学习上采样和逐元素跳跃融合这四个核心点。


---


## 8. 总结 Summary

- FCN 将分类网络中的全连接层替换为卷积层，使网络能够输出二维类别 score map。
- 语义分割要求对每个像素分类，模型最终输出 logits 的形状通常为 $(N, C, H, W)$。
- 下采样带来更强语义但损失位置细节，FCN 用转置卷积将预测恢复到输入大小。
- FCN-32s 只使用深层预测；FCN-16s 和 FCN-8s 融合 pool4、pool3 等浅层特征，边界更准确。
- 跳跃连接使用逐元素相加，融合前必须匹配类别通道和空间尺寸。
- 多类别分割常用 CrossEntropyLoss，标签通常为 $(N, H, W)$ 的类别索引，模型输出不要先经过 softmax。
- 训练中最容易出错的是图像与 mask 变换不同步、mask 使用错误插值方式、输出和标签尺寸不一致，以及错误处理 ignore index。


***********
